In [2]:
import rasterio
import richdem as rd
import numpy as np

dem_path = r"D:\wenqu\chapter1_2\chapter2\DEM\DEM_tif\site1b_dem_part1.tif"
slope_path = r"D:\wenqu\chapter1_2\chapter2\DEM\slope\site1b_slope.tif"
aspect_path = r"D:\wenqu\chapter1_2\chapter2\DEM\slope\site1b_aspect.tif"
tri_path = r"D:\wenqu\chapter1_2\chapter2\DEM\slope\site1b_tri.tif"
tpi_path = r"D:\wenqu\chapter1_2\chapter2\DEM\slope\site1b_tpi.tif"
twi_path = r"D:\wenqu\chapter1_2\chapter2\DEM\slope\site1b_twi.tif"

# --- 1. Load DEM as richdem object ---
with rasterio.open(dem_path) as src:
    dem_array = src.read(1).astype('float32')
    profile = src.profile
    cellsize_x = src.transform.a
    cellsize_y = -src.transform.e  # usually positive
    cellsize = (abs(cellsize_x) + abs(cellsize_y)) / 2

rdem = rd.rdarray(dem_array, no_data=profile.get("nodata", -9999))

# --- 2. Fill sinks (important for hydrology) ---
rdem_filled = rd.FillDepressions(rdem, in_place=False)

# --- 3. Slope & Aspect (degrees) ---
slope = rd.TerrainAttribute(rdem_filled, attrib='slope_degrees')
aspect = rd.TerrainAttribute(rdem_filled, attrib='aspect')

# --- 4. TRI & TPI ---
tri = rd.TerrainAttribute(rdem_filled, attrib='TRI')  # Terrain Ruggedness Index
tpi = rd.TerrainAttribute(rdem_filled, attrib='TPI')  # Topographic Position Index

# --- 5. Flow Direction & Flow Accumulation ---
fdir = rd.FlowDirectionD8(rdem_filled)          # D8 flow direction
fac = rd.FlowAccumulation(fdir, method='D8')    # Flow accumulation (number of cells)

# --- 6. Topographic Wetness Index (TWI) ---
# Convert slope to radians
slope_rad = np.deg2rad(slope)

# Avoid divide by zero
slope_rad[slope_rad <= 0] = np.nan

# Specific catchment area (a) ≈ flow_accumulation * cellsize
sca = (fac + 1) * cellsize  # +1 to avoid log(0)

twi = np.log(sca / np.tan(slope_rad))

# --- 7. Write outputs ---
def write_raster(path, array, profile):
    out_profile = profile.copy()
    out_profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)
    with rasterio.open(path, 'w', **out_profile) as dst:
        dst.write(array.astype('float32'), 1)

write_raster(slope_path, slope, profile)
write_raster(aspect_path, aspect, profile)
write_raster(tri_path, tri, profile)
write_raster(tpi_path, tpi, profile)
write_raster(twi_path, twi, profile)


Warning! No geotransform defined. Choosing a standard one! (Top left cell's top let corner at <0,0>; cells are 1x1.)
Warning! No geotransform defined. Choosing a standard one! (Top left cell's top let corner at <0,0>; cells are 1x1.)
Warning! No geotransform defined. Choosing a standard one! (Top left cell's top let corner at <0,0>; cells are 1x1.)
Warning! No geotransform defined. Choosing a standard one! (Top left cell's top let corner at <0,0>; cells are 1x1.)
Warning! No geotransform defined. Choosing a standard one! (Top left cell's top let corner at <0,0>; cells are 1x1.)


Exception: Invalid TerrainAttributes attribute. Valid attributes are: slope_riserun, slope_percentage, slope_degrees, slope_radians, aspect, curvature, planform_curvature, profile_curvature

In [1]:
import rasterio
import numpy as np
from scipy.ndimage import generic_filter
import pyflwdir

dem_path = r"D:\wenqu\chapter1_2\chapter2\DEM\DEM_tif\site1b_dem_part1.tif"

# -------------------------
# 1. Read DEM
# -------------------------
with rasterio.open(dem_path) as src:
    dem = src.read(1).astype('float32')
    profile = src.profile
    transform = src.transform
    res_x, res_y = src.res
    cellsize = res_x

# -------------------------
# 2. Slope and Aspect (numpy)
# -------------------------
dzdx = (np.roll(dem, -1, axis=1) - np.roll(dem, 1, axis=1)) / (2 * cellsize)
dzdy = (np.roll(dem, -1, axis=0) - np.roll(dem, 1, axis=0)) / (2 * cellsize)

slope = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))
aspect = np.degrees(np.arctan2(dzdy, -dzdx))
aspect = np.where(aspect < 0, 90 - aspect, 360 - aspect + 90)

# -------------------------
# 3. TRI (Terrain Ruggedness Index)
# -------------------------
def tri_func(window):
    center = window[4]
    return np.sqrt(np.mean((window - center) ** 2))

tri = generic_filter(dem, tri_func, size=3)

# -------------------------
# 4. TPI (Topographic Position Index)
# -------------------------
def tpi_func(window):
    center = window[4]
    neighbors = np.delete(window, 4)
    return center - np.mean(neighbors)

tpi = generic_filter(dem, tpi_func, size=5)

# -------------------------
# 5. Flow direction + Flow accumulation from pyflwdir
# -------------------------
flw = pyflwdir.from_dem(dem, transform=transform, latlon=False)

fac = flw.acc # flow accumulation (number of cells)

# -------------------------
# 6. TWI (Topographic Wetness Index)
# -------------------------
slope_rad = np.deg2rad(slope)
slope_rad[slope_rad <= 0] = np.nan

sca = (fac + 1) * cellsize   # specific catchment area

twi = np.log(sca / np.tan(slope_rad))

# -------------------------
# 7. Save outputs
# -------------------------
def write_raster(path, arr):
    out = profile.copy()
    out.update(dtype="float32", count=1, nodata=np.nan)
    with rasterio.open(path, "w", **out) as dst:
        dst.write(arr.astype("float32"), 1)

write_raster("slope.tif", slope)
write_raster("aspect.tif", aspect)
write_raster("tri.tif", tri)
write_raster("tpi.tif", tpi)
write_raster("twi.tif", twi)
write_raster("flowacc.tif", fac)


C:\Users\laral\AppData\Local\Temp\ipykernel_19752\679823732.py:24: RuntimeWarning: overflow encountered in square
  slope = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))


AttributeError: 'FlwdirRaster' object has no attribute 'acc'

In [5]:
import sys
print(sys.executable)


C:\Users\laral\Anaconda3\envs\wenqu_gpu\python.exe


In [1]:
import rasterio
import numpy as np
from scipy.ndimage import generic_filter
import pyflwdir

dem_path = r"D:\wenqu\chapter1_2\chapter2\DEM\DEM_tif\site6_dem.tif"

# -------------------------
# 1. Read DEM and clean nodata
# -------------------------
with rasterio.open(dem_path) as src:
    dem = src.read(1).astype("float64")  # float64 to avoid overflow
    profile = src.profile
    transform = src.transform
    res_x, res_y = src.res
    cellsize = res_x  # assuming square pixels
    nodata = src.nodata

# set nodata to NaN so we don't get crazy gradients at edges
if nodata is not None:
    dem = np.where(dem == nodata, np.nan, dem)

# -------------------------
# 2. Slope & aspect with np.gradient
# -------------------------
dzdy, dzdx = np.gradient(dem, cellsize, cellsize)
grad_mag = np.hypot(dzdx, dzdy)  # sqrt(dzdx^2 + dzdy^2)

slope = np.degrees(np.arctan(grad_mag))  # slope (degrees)

aspect = np.degrees(np.arctan2(dzdy, -dzdx))
# convert to 0–360, 0 = north
aspect = np.where(aspect < 0, 90.0 - aspect, 360.0 - aspect + 90.0)

slope[np.isnan(dem)] = np.nan
aspect[np.isnan(dem)] = np.nan

# -------------------------
# 3. TRI (Terrain Ruggedness Index)
# -------------------------
def tri_func(window):
    center = window[4]
    return np.sqrt(np.nanmean((window - center) ** 2))

tri = generic_filter(dem, tri_func, size=3, mode="nearest")
tri[np.isnan(dem)] = np.nan

# -------------------------
# 4. TPI (Topographic Position Index)
# -------------------------
def tpi_func(window):
    center = window[4]
    neighbors = np.delete(window, 4)
    return center - np.nanmean(neighbors)

tpi = generic_filter(dem, tpi_func, size=5, mode="nearest")
tpi[np.isnan(dem)] = np.nan

# -------------------------
# 5. Flow direction + upstream area (flow accumulation proxy)
# -------------------------
# pyflwdir needs numeric nodata, not NaN
dem_for_flw = dem.copy()
nodata_flw = -9999.0 if nodata is None else float(nodata)
dem_for_flw = np.where(np.isnan(dem_for_flw), nodata_flw, dem_for_flw)

flw = pyflwdir.from_dem(
    data=dem_for_flw,
    nodata=nodata_flw,
    transform=transform,
    latlon=False,
)

# ✅ THIS is the correct way to get "flow accumulation"
fac = flw.upstream_area(unit="cell")   # number of contributing cells
# or: fac_m2 = flw.upstream_area(unit="m2")  # contributing area in m²

fac = fac.astype("float64")
fac[fac <= 0] = np.nan

# -------------------------
# 6. TWI (Topographic Wetness Index)
#     TWI = ln( a / tan(slope) )
#     where a = specific catchment area
# -------------------------
slope_rad = np.deg2rad(slope)
slope_rad[slope_rad <= 0] = np.nan

# specific catchment area (in m): contributing cells * area / cell width
a_m2 = fac * (cellsize ** 2)   # each cell ~ cellsize² m²
sca = a_m2 / cellsize          # m² / m = m

twi = np.log(sca / np.tan(slope_rad))
twi[np.isinf(twi)] = np.nan

# -------------------------
# 7. Helper to save rasters
# -------------------------
def write_raster(path, arr):
    out = profile.copy()
    out.update(dtype="float32", count=1, nodata=np.nan)
    with rasterio.open(path, "w", **out) as dst:
        dst.write(arr.astype("float32"), 1)

# -------------------------
# 8. Save outputs
# -------------------------
write_raster(r"D:\wenqu\chapter1_2\chapter2\DEM\site6\slope.tif", slope)
write_raster(r"D:\wenqu\chapter1_2\chapter2\DEM\site6\aspect.tif", aspect)
write_raster(r"D:\wenqu\chapter1_2\chapter2\DEM\site6\tri.tif", tri)
write_raster(r"D:\wenqu\chapter1_2\chapter2\DEM\site6\tpi.tif", tpi)
write_raster(r"D:\wenqu\chapter1_2\chapter2\DEM\site6\flowacc_cells.tif", fac)  # number of contributing cells
write_raster(r"D:\wenqu\chapter1_2\chapter2\DEM\site6\twi.tif", twi)


C:\Users\laral\AppData\Local\Temp\ipykernel_29416\1836836525.py:43: RuntimeWarning: Mean of empty slice
  return np.sqrt(np.nanmean((window - center) ** 2))
C:\Users\laral\AppData\Local\Temp\ipykernel_29416\1836836525.py:54: RuntimeWarning: Mean of empty slice
  return center - np.nanmean(neighbors)
